<a href="https://colab.research.google.com/github/maximoalva/nlp-multidb-rag/blob/main/TP_NLP_MaximoAlva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo práctico final - Procesamiento del Lenguaje natural

---

Tecnicatura Universitaria en Inteligencia Artificial

Facultad de Ciencias Exactas, Ingeniería y Agrimensura

Universidad Nacional de Rosario

Máximo Alva

2026

# Preparación del entorno

Descargar e importar librerias.

In [ ]:
!pip -q install jedi sentence-transformers pymilvus milvus-lite langchain-text-splitters "grafitodb[viz]" matplotlib llama_index==0.12.39 rank_bm25 unidecode nltk langchain-ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.7/179.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
import json
from unidecode import unidecode
from typing import List, Optional
import matplotlib.pyplot as plt

from langchain_text_splitters import MarkdownHeaderTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from pymilvus import MilvusClient
import shutil
import logging
import sqlite3
from grafito import GrafitoDatabase
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from llama_index.core.schema import TextNode, NodeWithScore, Document as LlamaDocument
from llama_index.core import SimpleDirectoryReader
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


Instalar Ollama

In [ ]:
# Install zstd, a required dependency for Ollama extraction
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

# The 'cuda-drivers' installation previously failed and is not essential for basic Ollama functionality (CPU).
# Removing related lines to avoid errors and simplify the installation process.
# !echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
# !sudo apt-get update && sudo apt-get install -y cuda-drivers

# Verify Ollama installation. Using full path for robustness.
# This confirms the binary is available at the expected location.
!/usr/local/bin/ollama --version

# This environment variable setting is typically for NVIDIA GPU libraries.
# Keeping it if the user intends GPU usage later, but it's not the root cause of 'ollama command not found'.
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,028 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubunt

In [ ]:
# Iniciar el servidor en segundo plano
!nohup /usr/local/bin/ollama serve &
# Extraer modelo 'qwen2.5'
!/usr/local/bin/ollama pull qwen2.5
# Instalar e importar paquete Ollama
!pip install ollama
import ollama

nohup: appending output to 'nohup.out'



Descargar fuentes de información.

In [ ]:
# Clonar repo
!git clone https://github.com/maximoalva/nlp-multidb-rag.git
# Mover 'fuentes_de_informacion' al directorio principal
!mv nlp-multidb-rag/fuentes_de_informacion /content/
# Borrar repo
!rm -rf nlp-multidb-rag

In [ ]:
#!gdown --fuzzy "https://drive.google.com/file/d/1FfzaP6EZBtD4QDpjQ-TU4MQ0KQ3N5zbx/view?usp=sharing"
#!unzip -q fuentes_de_informacion.zip
#!rm fuentes_de_informacion.zip

Downloading...
From: https://drive.google.com/uc?id=1FfzaP6EZBtD4QDpjQ-TU4MQ0KQ3N5zbx
To: /content/fuentes_de_informacion.zip
100% 3.81M/3.81M [00:00<00:00, 33.9MB/s]


# Ejercicio 1: RAG

##  Base de datos vectorial

### Extracción de texto

In [ ]:
BASE_DIR = Path('/content/vector_store_demo')
BASE_DIR.mkdir(parents=True, exist_ok=True)

MANUALES_PRODUCTOS = Path('/content/fuentes_de_informacion/manuales_productos')
RESEÑAS_USUARIOS = Path('/content/fuentes_de_informacion/resenas_usuarios')

RESET_STORES = True

# Configurar TextSplitter respetando secciones
headers_to_split_on = [("#", "Título"), ("##", "Sección"), ("###", "Subsección")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Acumulador de documentos
documents = []
doc_id_counter = 1


# Procesamiento de manuales
for path in list(MANUALES_PRODUCTOS.glob('**/*.md')):
    with open(path, 'r', encoding='utf-8') as f:
        contenido = f.read()

    # Elimina negritas y cursivas
    contenido = re.sub(r'(\*\*|__|\*|_)', '', contenido)

    # Extraer id_producto
    match_id_producto = re.search(r'(P\d+)', path.name)
    id_producto = match_id_producto.group(1)

    # Extraer marca
    match_marca = re.search(r'Marca:\s*([^\n\|]+)', contenido)
    marca = match_marca.group(1).strip()

    # Extraer categoría
    match_categoria = re.search(r'Categoría:\s*([^\n\-\|]+)', contenido)
    categoria = match_categoria.group(1).strip().lower()

    # Limpiar nombre del producto
    nombre = path.stem.replace('manual_', '')
    nombre = re.sub(r'P\d+_*', '', nombre).replace('_', ' ').strip()

    # Eliminar índice para evitar ruido semántico
    contenido = re.sub(r'## Índice\b.*?(\n##\s)', r'\1', contenido, flags=re.DOTALL)

    # Fragmentar
    fragmentos = md_splitter.split_text(contenido)

    # Guardar cada fragmento con cabecera
    for fr in fragmentos:
        # Extraemos los metadatos que generó el splitter
        jerarquia_seccion = " > ".join([v for k, v in fr.metadata.items()])
        # Armamos cabecera
        fr_enriquecido = f"Producto: {nombre} {id_producto} Marca: {marca} | {jerarquia_seccion} | {fr.page_content}"

        documents.append({
            'id': doc_id_counter,
            'categoria': categoria,
            'text': fr_enriquecido,
            'id_producto': id_producto
        })

        doc_id_counter += 1


# Procesamiento de reseñas
for path in list(RESEÑAS_USUARIOS.glob('**/*.txt')):
    with open(path, 'r', encoding='utf-8') as f:
        contenido = f.read()

    # Separar cabecera del cuerpo
    cabecera, cuerpo = map(str.strip, contenido.split('\n\n', 1))

    # Extraer metadatos
    match_prod = re.search(r'Producto:\s*(.+?)\s*\((P\d+)\)', cabecera)
    nombre = match_prod.group(1).strip()
    id_producto = match_prod.group(2)

    match_puntaje = re.search(r'Puntaje:\s*(\d+)', cabecera)
    puntaje = int(match_puntaje.group(1))

    # Armar texto y guardar
    fr_enriquecido = f"Reseña de usuario ({puntaje}/5 estrellas) para el producto {nombre} {id_producto}: {cuerpo}"

    documents.append({
        'id': doc_id_counter,
        'categoria': 'resena', # Categoría dura para filtros
        'text': fr_enriquecido,
        'id_producto': id_producto
    })
    doc_id_counter += 1

### Generación de embeddings

In [ ]:
# Inicializar modelo de embeddings
embedding_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Generar embeddings
texts = [doc['text'] for doc in documents]
embeddings = embedding_model.encode(texts, normalize_embeddings=True).astype('float32')
DIM = embeddings.shape[1]

for doc, vector in zip(documents, embeddings):
    doc['vector'] = vector.tolist()

def embed_query(query: str) -> np.ndarray:
    return embedding_model.encode([query], normalize_embeddings=True).astype('float32')[0]

def show_results(rows):
    return pd.DataFrame(rows, columns=['rank', 'score', 'categoria', 'text'])

print(f'Documentos: {len(documents)}\nDimension de embeddings: {DIM}\nDirectorio local: {BASE_DIR}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documentos: 5979
Dimension de embeddings: 384
Directorio local: /content/vector_store_demo


### Milvus Lite

In [ ]:
# Silenciar el error interno de gRPC en Milvus Lite
logging.getLogger("grpc._server").setLevel(logging.CRITICAL)

milvus_path = BASE_DIR / 'milvus_lite.db'
if RESET_STORES and milvus_path.exists():
    if milvus_path.is_dir():
        shutil.rmtree(milvus_path)
    else:
        milvus_path.unlink()

mclient = MilvusClient(uri=str(milvus_path))
milvus_collection = 'documents'

if mclient.has_collection(milvus_collection):
    mclient.drop_collection(milvus_collection)

mclient.create_collection(
    collection_name=milvus_collection,
    dimension=DIM,
    metric_type='COSINE',
    consistency_level='Strong',
)

mclient.insert(
    collection_name=milvus_collection,
    data=[
        {
            'id': doc['id'],
            'vector': doc['vector'],
            'text': doc['text'],
            'categoria': doc['categoria'],
            'id_producto': doc['id_producto']
        }
        for doc in documents
    ],
)

{'insert_count': 5979, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 21

### Interfaz

In [ ]:
def buscar_db_vectorial(query: str, k: int = 3, categoria_filtro: str = None) -> pd.DataFrame:
    """
    Recibe consulta + filtro y devuelve los K fragmentos más similares.
    """
    # Convertir consulta a vector
    query_vector = embed_query(query)

    # Si pasaron una categoría, armamos la expresión de filtro para Milvus
    filter_expr = ""
    if categoria_filtro:
      filter_expr = f"categoria == '{categoria_filtro.lower()}'"

    # Buscar en Milvus
    milvus_hits = mclient.search(
        collection_name=milvus_collection,
        data=[query_vector.tolist()],
        limit=k,
        filter=filter_expr,
        output_fields=['text', 'categoria'],
        search_params={'metric_type': 'COSINE'},
    )[0]

    # Devolver df con resultados
    return show_results([
        (i + 1, float(hit['distance']), hit['entity']['categoria'], hit['entity']['text'])
        for i, hit in enumerate(milvus_hits)
    ])

In [ ]:
# Test 1: ¿Cómo uso mi licuadora para hacer smoothies?
display(buscar_db_vectorial(query="¿Cómo uso mi licuadora para hacer smoothies?"))

,rank,score,categoria,text
0,1,0.484122,cocina,Producto: Deluxe Cafetera 2024 P0120 Marca: Ho...
1,2,0.497932,cocina,Producto: Compacto Licuadora P0004 Marca: Chef...
2,3,0.509005,cocina,Producto: Eco Mixer II P0023 Marca: CookElite ...


## Base de datos tabular

### SQL

In [ ]:
# Conexión a base de datos SQL en memoria
conn = sqlite3.connect(':memory:')

# Cargar CSVs en Pandas y volcarlos a SQL
CSV_DIR = Path('/content/fuentes_de_informacion')

archivos = {
    'productos': 'productos.csv',
    'ventas': 'ventas_historicas.csv',
    'tickets': 'tickets_soporte.csv',
    'inventario': 'inventario_sucursales.csv',
    'vendedores': 'vendedores.csv',
    'devoluciones': 'devoluciones.csv'
}

for nombre_tabla, archivo in archivos.items():
    ruta = CSV_DIR / archivo
    df = pd.read_csv(ruta)
    # Cargar el df directamente como una tabla SQL
    df.to_sql(nombre_tabla, conn, index=False, if_exists='replace')
    print(f"Tabla '{nombre_tabla}' cargada exitosamente.")

# Recopilar información relevante
# Valores únicos de la tabla de productos para dárselos al LLM
df_prod_info = pd.read_sql("SELECT DISTINCT categoria, marca FROM productos", conn)
categorias_unicas = df_prod_info['categoria'].dropna().unique().tolist()
marcas_unicas = df_prod_info['marca'].dropna().unique().tolist()

# Precios para que el LLM sepa los rangos
precio_min = pd.read_sql("SELECT MIN(precio_usd) as min FROM productos", conn).iloc[0]['min']
precio_max = pd.read_sql("SELECT MAX(precio_usd) as max FROM productos", conn).iloc[0]['max']

Tabla 'productos' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'tickets' cargada exitosamente.
Tabla 'inventario' cargada exitosamente.
Tabla 'vendedores' cargada exitosamente.
Tabla 'devoluciones' cargada exitosamente.


### Contexto

In [ ]:
# Construir string de formato para el LLM
contexto_tabular_llm = f"""
Estás actuando como un asistente de análisis de datos y desarrollador SQL.
A continuación se presenta el esquema de la base de datos relacional de la empresa:

1. Tabla 'productos': id_producto, nombre, categoria, subcategoria, marca, precio_usd, stock, color, potencia_w, capacidad, voltaje, peso_kg, garantia_meses, descripcion
2. Tabla 'ventas': id_venta, fecha, hora, id_producto, nombre_producto, id_vendedor, nombre_vendedor, sucursal, cantidad, precio_unitario, descuento_pct, total, metodo_pago, cliente_nombre, cliente_provincia
3. Tabla 'inventario': id_inventario, sucursal, id_producto, nombre_producto, categoria, marca, stock_sucursal, stock_minimo, stock_maximo, precio_sucursal, ultima_reposicion, proveedor, pasillo, estado
4. Tabla 'tickets': id_ticket, fecha_apertura, id_venta, id_producto, nombre_producto, cliente_nombre, cliente_provincia, tipo_problema, descripcion, severidad, categoria, estado, vendedor_asignado, sucursal, fecha_resolucion, dias_resolucion, garantia_valida
5. Tabla 'devoluciones': id_devolucion, id_venta, fecha_devolucion, id_producto, nombre_producto, cliente_nombre, motivo, descripcion_cliente, estado, monto_venta, monto_reembolso, metodo_reembolso, fecha_reembolso, observaciones
6. Tabla 'vendedores': id_vendedor, nombre, apellido, email, telefono, sucursal, fecha_ingreso, nivel, comision_pct, activo

Valores de referencia en 'productos':
- Categorías válidas: {categorias_unicas}
- Marcas válidas: {marcas_unicas}
- Rango de precios: ${precio_min} a ${precio_max}

REGLAS CRÍTICAS PARA TUS CONSULTAS SQL:
1. Devuelve ÚNICAMENTE la consulta SQL válida (en dialecto SQLite). No des explicaciones, no uses formato markdown (
```sql). SOLO el código puro.
2. Si la consulta menciona un código como 'P0017', SIEMPRE debes buscar en la columna 'id_producto' (ej: WHERE id_producto = 'P0017').
3. Si buscas por nombre, usa SIEMPRE operadores LIKE con comodines '%' en lugar de igualdades estrictas (ej: WHERE nombre LIKE '%batidora%'), ya que los nombres pueden no ser exactos.

Tu tarea es recibir una pregunta del usuario y devolver ÚNICAMENTE una consulta SQL válida (en dialecto SQLite) que responda a esa pregunta.
No des explicaciones. Devuelve SOLO el código SQL.
"""

### Interfaz

In [ ]:
def buscar_db_tabular(nl_query: str) -> pd.DataFrame:
    """
    Recibe consulta en lenguaje natural, la traduce a SQL (vía LLM),
    ejecuta la consulta en la base de datos y devuelve un DataFrame.
    """
    try:
        # LLamar al LLM con el contexto y la pregunta
        response = ollama.chat(
            model='qwen2.5',
            messages=[
                {"role": "system", "content": contexto_tabular_llm},
                {"role": "user", "content": f"Pregunta: {nl_query}"}
            ]
        )

        # Extraer y limpiar el texto generado por si el modelo incluyó markdown sin querer
        texto_generado = response['message']['content'].strip()
        if "```" in texto_generado:
            # Nos quedamos con lo que está adentro de las comillas
            sql_generado = texto_generado.split("```")[1]
            # Borrar las etiquetas de lenguaje si quedaron pegadas
            sql_generado = sql_generado.replace("sqlite", "").replace("sql", "").strip()
        else:
            sql_generado = texto_generado

        # Ejecutar la consulta SQL generada en la base de datos
        resultados = pd.read_sql_query(sql_generado, conn)

        return resultados, sql_generado

    except Exception as e:
        # Si el LLM arma mal la query o falla la conexión, devolvemos error en formato DataFrame
        return pd.DataFrame({"Error sistema tabular": [str(e)]}), "Error al generar SQL"

## Base de datos de grafos

### GrafitoDB

In [ ]:
# Base de datos de grafos en memoria (sin servidor, respaldada por SQLite).
# cypher_max_hops define el límite por defecto para caminos de longitud variable.
db_grafito = GrafitoDatabase(':memory:', cypher_max_hops=6)

# Cargar datos
# JSON con FAQs
with open('/content/fuentes_de_informacion/faqs.json', 'r', encoding='utf-8') as f:
    df_faqs = pd.DataFrame(json.load(f))
# CSV con productos
df_productos = pd.read_csv('/content/fuentes_de_informacion/productos.csv')

# Crear nodos
nodos_guardados = {}
# Nodos de categorías (CSV)
for cat in df_productos['categoria'].dropna().unique():
    nodo = db_grafito.create_node(labels=['Categoria'], properties={'nombre': cat.lower()})
    nodos_guardados[f"CAT_{cat}"] = nodo.id

# Nodos de productos (CSV)
for _, row in df_productos.iterrows():
    nodo = db_grafito.create_node(labels=['Producto'], properties={'id': row['id_producto'], 'nombre': row['nombre']})
    nodos_guardados[f"PROD_{row['id_producto']}"] = nodo.id

# Nodos de temas de FAQ (JSON)
for tema in df_faqs['categoria'].dropna().unique():
    nodo = db_grafito.create_node(labels=['TemaFAQ'], properties={'nombre': tema.lower()})
    nodos_guardados[f"TEMA_{tema}"] = nodo.id

# Nodos de FAQs (JSON)
for _, row in df_faqs.iterrows():
    nodo = db_grafito.create_node(labels=['FAQ'], properties={'id': row['id_faq'], 'pregunta': row['pregunta']})
    nodos_guardados[f"FAQ_{row['id_faq']}"] = nodo.id


# DataFrame de relaciones
relaciones_prod_cat = pd.DataFrame({
    'origen': ["PROD_" + str(x) for x in df_productos['id_producto']],
    'relacion': 'PERTENECE_A',
    'destino': ["CAT_" + str(x) for x in df_productos['categoria']]
})

relaciones_prod_faq = pd.DataFrame({
    'origen': ["PROD_" + str(x) for x in df_faqs['id_producto']],
    'relacion': 'TIENE_FAQ',
    'destino': ["FAQ_" + str(x) for x in df_faqs['id_faq']]
})

relaciones_faq_tema = pd.DataFrame({
    'origen': ["FAQ_" + str(x) for x in df_faqs['id_faq']],
    'relacion': 'TRATA_SOBRE',
    'destino': ["TEMA_" + str(x) for x in df_faqs['categoria']]
})

df_relaciones = pd.concat([relaciones_prod_cat, relaciones_prod_faq, relaciones_faq_tema], ignore_index=True).dropna()


# Insertar relaciones (aristas)
relaciones_creadas = 0
for _, row in df_relaciones.iterrows():
    # Verificamos que ambos nodos existan en nuestro diccionario
    if row['origen'] in nodos_guardados and row['destino'] in nodos_guardados:
        id_orig = nodos_guardados[row['origen']]
        id_dest = nodos_guardados[row['destino']]
        db_grafito.create_relationship(id_orig, id_dest, row['relacion'])
        relaciones_creadas += 1

print(f"Nodos: {len(nodos_guardados)}\nRelaciones: {relaciones_creadas}")

Nodos: 3309
Relaciones: 6300


### Contexto

In [ ]:
contexto_grafito_llm = """
Eres un experto en bases de datos de grafos. Tu tarea es traducir lenguaje natural a código Cypher.
El esquema de la base de datos es el siguiente:
- (Producto {id: '...', nombre: '...'})
- (Categoria {nombre: '...'})
- (FAQ {id: '...', pregunta: '...'})
- (TemaFAQ {nombre: '...'})

Relaciones válidas:
- (Producto)-[:PERTENECE_A]->(Categoria)
- (Producto)-[:TIENE_FAQ]->(FAQ)
- (FAQ)-[:TRATA_SOBRE]->(TemaFAQ)

Regla crítica:
1. Devuelve ÚNICAMENTE la consulta Cypher válida.
2. No uses formato markdown de código (```).
3. Convierte las categorías y temas a minúsculas en tus validaciones (ej: {nombre: 'cocina'}).

Ejemplo de usuario: "¿Qué productos están relacionados con la categoría Cocina?"
Respuesta esperada:
MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'cocina'}) RETURN p.nombre
"""

### Interfaz

In [ ]:
def buscar_db_grafos(nl_query: str) -> list:
    """
    Recibe consulta en lenguaje natural, la traduce a Cypher (vía LLM),
    ejecuta la consulta en GrafitoDB y devuelve los resultados.
    """
    try:
        # LLamar al LLM con el contexto y la pregunta
        response = ollama.chat(
            model='qwen2.5',
            messages=[
                {"role": "system", "content": contexto_grafito_llm},
                {"role": "user", "content": f"Pregunta: {nl_query}"}
            ]
        )

        # Extraer y limpiar el texto generado por si el modelo incluyó markdown sin querer
        texto_generado = response['message']['content'].strip()
        # Nos quedamos con lo que está adentro de las comillas
        if "```" in texto_generado:
            cypher_generado = texto_generado.split("```")[1]
            # Borrar las etiquetas de lenguaje si quedaron pegadas
            cypher_generado = cypher_generado.replace("cypher", "").strip()
        else:
            cypher_generado = texto_generado

        # Ejecución de consulta en GrafitoDB
        resultados = list(db_grafito.execute(cypher_generado))

        return resultados, cypher_generado

    except Exception as e:

        return [{"Error sistema de grafos": str(e)}], "Error al generar Cypher"

##  Clasificador de intención avanzado

### Clasificador entrenado

In [ ]:
# Dataset sintético
datos_sinteticos = [
    # VECTORIAL (Manuales, usos, opiniones, especificaciones en texto libre)
    {"pregunta": "¿Cómo limpio el filtro de la licuadora?", "intencion": "vectorial"},
    {"pregunta": "Quiero saber las opiniones de la cafetera", "intencion": "vectorial"},
    {"pregunta": "¿Qué dicen las reseñas del caloventor?", "intencion": "vectorial"},
    {"pregunta": "Mostrame las instrucciones para armar el ventilador", "intencion": "vectorial"},
    {"pregunta": "¿Por qué hace ruido mi batidora?", "intencion": "vectorial"},
    {"pregunta": "Solución de problemas si no enciende la pava eléctrica", "intencion": "vectorial"},
    {"pregunta": "¿Qué opinan los usuarios de este producto?", "intencion": "vectorial"},
    {"pregunta": "¿Cómo uso mi licuadora para hacer smoothies?", "intencion": "vectorial"},
    {"pregunta": "Mejor forma de mantener la sandwichera limpia", "intencion": "vectorial"},
    {"pregunta": "¿Qué cubre la garantía de la heladera?", "intencion": "vectorial"},
    # TABULAR (Precios, filtros exactos, rangos, conteos, sucursales)
    {"pregunta": "¿Cuáles son las licuadoras de menos de $200?", "intencion": "tabular"},
    {"pregunta": "Mostrame los productos de la marca ChefMaster", "intencion": "tabular"},
    {"pregunta": "¿Cuántos televisores hay en stock en la sucursal Centro?", "intencion": "tabular"},
    {"pregunta": "Listar productos con precio mayor a 50000 pesos", "intencion": "tabular"},
    {"pregunta": "¿Qué vendedores tienen una comisión mayor al 10%?", "intencion": "tabular"},
    {"pregunta": "Quiero ver las devoluciones del último mes", "intencion": "tabular"},
    {"pregunta": "¿Cuál es el producto más barato de la categoría cocina?", "intencion": "tabular"},
    {"pregunta": "Filtrar los tickets de soporte en estado abierto", "intencion": "tabular"},
    {"pregunta": "Dame el total de ventas del vendedor Juan Perez", "intencion": "tabular"},
    {"pregunta": "Ordenar las cafeteras por precio de menor a mayor", "intencion": "tabular"},
    # GRAFOS (Relaciones, categorías, accesorios, compatibilidad)
    {"pregunta": "¿Qué productos están relacionados con la categoría Cocina?", "intencion": "grafos"},
    {"pregunta": "¿Qué accesorios son compatibles con la licuadora P0004?", "intencion": "grafos"},
    {"pregunta": "Mostrame todos los productos que pertenecen a climatización", "intencion": "grafos"},
    {"pregunta": "¿Cuáles son las preguntas frecuentes asociadas al televisor?", "intencion": "grafos"},
    {"pregunta": "¿Con qué otros aparatos comparte repuestos la vinoteca?", "intencion": "grafos"},
    {"pregunta": "¿Qué temas se tratan en las FAQs de la batidora?", "intencion": "grafos"},
    {"pregunta": "Listar los productos conectados a la categoría limpieza", "intencion": "grafos"},
    {"pregunta": "¿Qué artículos están vinculados con la marca Samsung?", "intencion": "grafos"},
    {"pregunta": "Relaciones de repuestos para el mixer", "intencion": "grafos"},
    {"pregunta": "¿Qué productos están en la misma categoría que el horno eléctrico?", "intencion": "grafos"}
]

df_sintetico = pd.DataFrame(datos_sinteticos)

In [ ]:
# Preparado de datos (embedding semántico)
# Usamos nuestro embedding_model para vectorizar nuestras preguntas
X = embedding_model.encode(df_sintetico['pregunta'].tolist(), normalize_embeddings=True)
y = df_sintetico['intencion']
# División 80/20 del dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Entrenamiento del modelo clasificador
modelo_logreg = LogisticRegression(random_state=42, max_iter=1000)
modelo_logreg.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
def clasificador_ml(query: str) -> str:
    """
    Recibe una consulta, la vectoriza usando embeddings semánticos
    y predice su intención con un modelo de regresión logística
    Devuelve: 'vectorial', 'tabular' o 'grafos'.
    """
    # Convertir la consulta al mismo espacio vectorial
    query_vector = embedding_model.encode([query], normalize_embeddings=True)

    # Predecir con Regresión Logística
    prediccion = modelo_logreg.predict(query_vector)

    return prediccion[0]

### Clasificador basado en LLM

In [ ]:
def clasificador_llm(query: str) -> str:
    """
    Clasifica la intención de la consulta usando un LLM local (gemma3:1b) con Few-Shot Prompting.
    Devuelve: 'vectorial', 'tabular' o 'grafos'.
    """
    # Prompt de sistema Few-Shot
    prompt_sistema = """
    Eres el clasificador de intención de un agente inteligente de una tienda de electrodomésticos.
    Tu única tarea es leer la consulta del usuario y clasificarla en EXACTAMENTE UNA de estas tres categorías:

    1. "vectorial": Preguntas sobre cómo usar un producto, funcionamiento, opiniones, reseñas o manuales.
    2. "tabular": Preguntas que requieran aplicar filtros exactos (precios, marcas, límites numéricos) o datos de bases de datos.
    3. "grafos": Preguntas que busquen relaciones entre entidades (ej: qué productos pertenecen a una categoría, accesorios compatibles, vinculaciones).

    Aquí tienes ejemplos de cómo debes responder:

    Consulta: "¿Cómo limpio el filtro de la aspiradora?"
    Respuesta: vectorial

    Consulta: "Mostrar televisores por debajo de 500 dólares"
    Respuesta: tabular

    Consulta: "¿Qué productos pertenecen a la categoría climatización?"
    Respuesta: grafos

    Consulta: "Quiero saber si la cafetera tiene buenas opiniones de la gente"
    Respuesta: vectorial

    REGLA CRÍTICA: Debes responder ÚNICAMENTE con la palabra de la categoría (vectorial, tabular o grafos). No agregues saludos, explicaciones, ni puntos finales.
    """

    # Llamar al LLM
    try:
        response = ollama.chat(
            model='qwen2.5',
            messages=[
                {'role': 'system', 'content': prompt_sistema},
                {'role': 'user', 'content': f"Consulta: '{query}'\nRespuesta:"}
            ]
        )

        # Extraemos el texto generado
        texto_generado = response['message']['content'].strip().lower()

        # Validación con raíces
        if 'vec' in texto_generado:
            return 'vectorial'
        elif 'tab' in texto_generado:
            return 'tabular'
        elif 'graf' in texto_generado:
            return 'grafos'
        else:
            # Fallback seguro en caso de que el modelo devuelva algo incomprensible
            print(f"[WARNING] Clasificación ambigua. El modelo respondió: {texto_generado}")
            return 'vectorial'

    except Exception as e:
        print(f"Error de conexión con Ollama: {e}")
        return 'vectorial'

### Evaluación

In [ ]:
# Pruebas con prompts de verificación
prompts = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras de menos de $200?",
    "¿Qué opinan los usuarios de esta cafetera?",
    "Quiero una licuadora con buenas reseñas",
    "¿Qué productos están relacionados con la categoría Cocina?"
]

print("Clasificador ML")
for p in prompts:
    intencion = clasificador_ml(p)
    print(f"Pregunta: {p}")
    print(f"Intención detectada: -> {intencion}\n")

print("\nClasificador LLM")
for p in prompts:
    intencion = clasificador_llm(p)
    print(f"Pregunta: {p}")
    print(f"Intención detectada: -> {intencion}\n")

Clasificador ML
Pregunta: ¿Cómo uso mi licuadora para hacer smoothies?
Intención detectada: -> vectorial

Pregunta: ¿Cuáles son las licuadoras de menos de $200?
Intención detectada: -> tabular

Pregunta: ¿Qué opinan los usuarios de esta cafetera?
Intención detectada: -> vectorial

Pregunta: Quiero una licuadora con buenas reseñas
Intención detectada: -> tabular

Pregunta: ¿Qué productos están relacionados con la categoría Cocina?
Intención detectada: -> grafos


Clasificador LLM
Pregunta: ¿Cómo uso mi licuadora para hacer smoothies?
Intención detectada: -> vectorial

Pregunta: ¿Cuáles son las licuadoras de menos de $200?
Intención detectada: -> tabular

Pregunta: ¿Qué opinan los usuarios de esta cafetera?
Intención detectada: -> vectorial

Pregunta: Quiero una licuadora con buenas reseñas
Intención detectada: -> vectorial

Pregunta: ¿Qué productos están relacionados con la categoría Cocina?
Intención detectada: -> grafos



In [ ]:
# Métricas
preguntas_test = df_sintetico.loc[y_test.index, 'pregunta'].tolist()
etiquetas_reales = y_test.tolist()

# Clasificador ML
y_pred_ml = [clasificador_ml(p) for p in preguntas_test]
print("Clasificador ML")
print(classification_report(y_test, y_pred_ml))

# Clasificador LLM
print("Clasificador LLM")
y_pred_llm = [clasificador_llm(p) for p in preguntas_test]
print(classification_report(y_test, y_pred_llm))

Clasificador ML
              precision    recall  f1-score   support

      grafos       1.00      1.00      1.00         2
     tabular       0.67      1.00      0.80         2
   vectorial       1.00      0.50      0.67         2

    accuracy                           0.83         6
   macro avg       0.89      0.83      0.82         6
weighted avg       0.89      0.83      0.82         6

Clasificador LLM
              precision    recall  f1-score   support

      grafos       1.00      0.50      0.67         2
     tabular       1.00      1.00      1.00         2
   vectorial       0.67      1.00      0.80         2

    accuracy                           0.83         6
   macro avg       0.89      0.83      0.82         6
weighted avg       0.89      0.83      0.82         6



##  Pipeline de Recuperación (Retrieval)

### Buscador BM25

In [ ]:
class BM25Searcher:
    """
    Implementación de búsqueda BM25 compatible con LlamaIndex
    """
    def __init__(self, input_dir: str = None, documents: List[LlamaDocument] = None, language: str = 'spanish'):
        """
        Inicializa el buscador BM25.

        Args:
            input_dir: Directorio de entrada para cargar documentos (opcional)
            documents: Lista de documentos de LlamaIndex (opcional)
            language: Idioma para el tokenizador ('spanish' o 'english')
        """
        self.language = language

        # Descarga recursos necesarios de NLTK
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            nltk.download('punkt')

        # Cargar documentos
        if input_dir:
            self.documents = SimpleDirectoryReader(input_dir=input_dir).load_data()
        elif documents:
            self.documents = documents
        else:
            raise ValueError("Debe proporcionar input_dir o documents")

        # Convertir Documents a TextNodes para compatibilidad
        self.text_nodes = []
        for i, doc in enumerate(self.documents):
            text_node = TextNode(
                text=doc.text,
                metadata=doc.metadata if hasattr(doc, 'metadata') else {},
                id_=f"bm25_node_{i}"
            )
            self.text_nodes.append(text_node)

        # Preprocesamiento y tokenización de documentos
        self.corpus = [self._preprocess_text(node.text) for node in self.text_nodes]
        self.tokenized_corpus = [self._tokenize(text) for text in self.corpus]

        # Inicializar BM25
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def _preprocess_text(self, text: str) -> str:
        """
        Preprocesa el texto para mejorar la búsqueda.

        Args:
            text: Texto a preprocesar

        Returns:
            Texto preprocesado
        """
        # Convertir a minúsculas
        text = text.lower()

        # Eliminar acentos
        text = unidecode(text)

        # Eliminar caracteres especiales pero mantener espacios
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

        # Eliminar espacios múltiples
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def _tokenize(self, text: str) -> List[str]:
        """
        Tokeniza el texto usando NLTK.

        Args:
            text: Texto a tokenizar

        Returns:
            Lista de tokens
        """
        return word_tokenize(text, language=self.language)

    def retrieve(self, query: str, top_k: int = 5) -> List[NodeWithScore]:
        """
        Recupera los documentos más relevantes para una consulta.

        Args:
            query: Consulta de búsqueda
            top_k: Número de documentos a recuperar

        Returns:
            Lista de NodeWithScore compatible con LlamaIndex y rerankers
        """
        # Preprocesar y tokenizar la consulta
        processed_query = self._preprocess_text(query)
        tokenized_query = self._tokenize(processed_query)

        # Obtener scores BM25
        scores = self.bm25.get_scores(tokenized_query)

        # Obtener los índices de los top_k documentos
        top_indices = np.argsort(scores)[-top_k:][::-1]

        # Crear lista de resultados en formato LlamaIndex usando TextNodes
        results = []
        for idx in top_indices:
            if scores[idx] > 0:  # Solo incluir documentos con score positivo
                node = NodeWithScore(
                    node=self.text_nodes[idx],  # Usar TextNode en lugar de Document
                    score=float(scores[idx])
                )
                results.append(node)

        return results

### Reranker

In [ ]:
class Reranker:
    """
    Clase encargada de reordenar los nodos usando un Cross-Encoder multilingüe.
    """
    def __init__(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'):
        """
        Inicializa el reranker con un modelo cross-encoder.
        """
        self.model = CrossEncoder(model_name)

    def rerank(self, query: str, nodes: List[NodeWithScore], top_k: int = None) -> List[NodeWithScore]:
        """
        Reordena los nodos recuperados usando el cross-encoder.
        """
        if not nodes:
            return nodes

        pairs = [(query, node.node.text) for node in nodes]
        scores = self.model.predict(pairs)
        scored_nodes = list(zip(scores, nodes))
        scored_nodes.sort(key=lambda x: x[0], reverse=True)

        if top_k:
            scored_nodes = scored_nodes[:top_k]

        return [node for _, node in scored_nodes]

### Búsqueda híbrida

In [ ]:
# Transformar lista de diccionarios en LlamaDocuments
documents_llama = []
for doc in documents:
    text = doc.get('text', '')
    metadata = {k: v for k, v in doc.items() if k != 'text'}

    nuevo_doc = LlamaDocument(text=text, metadata=metadata)
    documents_llama.append(nuevo_doc)

# Inicializar motores
bm25_searcher = BM25Searcher(documents=documents_llama)
reranker = Reranker()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# Función puente para conectar Milvus con el ReRanker
def buscar_milvus(query: str, top_k: int = 5) -> list:
    vector_query = embed_query(query)

    resultados = mclient.search(
        collection_name=milvus_collection,
        data=[vector_query.tolist()],
        limit=top_k,
        output_fields=['text']
    )

    nodos_milvus = []
    if resultados and len(resultados[0]) > 0:
        for hit in resultados[0]:
            nodo = TextNode(text=hit['entity']['text'])
            nodo_con_score = NodeWithScore(node=nodo, score=hit['distance'])
            nodos_milvus.append(nodo_con_score)
    return nodos_milvus

In [ ]:
# Búsqueda híbrida
def buscar_db_vectorial_hibrida(query: str, k_final: int = 3) -> list:
    vector_nodes = buscar_milvus(query, top_k=5)
    bm25_nodes = bm25_searcher.retrieve(query, top_k=5)

    textos_vistos = set()
    nodos_unicos = []

    for nodo in (vector_nodes + bm25_nodes):
        clave_texto = " ".join(nodo.node.text.strip().lower().split()[:50])
        if clave_texto not in textos_vistos:
            textos_vistos.add(clave_texto)
            nodos_unicos.append(nodo)

    nodos_finales = reranker.rerank(query, nodos_unicos, top_k=k_final)

    return [(nodo.node.text, nodo.score) for nodo in nodos_finales]

### Enrutador

In [ ]:
def pipeline_recuperacion(query: str):
    intencion = clasificador_llm(query)

    if intencion == 'vectorial':
        return buscar_db_vectorial_hibrida(query), intencion
    elif intencion == 'tabular':
        return buscar_db_tabular(query), intencion
    elif intencion == 'grafos':
        return buscar_db_grafos(query), intencion
    else:
        return "Intención no reconocida.", intencion

## Checkpoint de Recuperación

In [ ]:
prompts = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras de menos de $200?",
    "¿Qué opinan los usuarios de esta cafetera?",
    "Quiero una licuadora con buenas reseñas",
    "¿Qué productos están relacionados con la categoría Cocina?"
]

for query in prompts:
    print(f"🗣️ USUARIO: '{query}'")

    resultados, intencion = pipeline_recuperacion(query)
    print(f"🎯 INTENCIÓN DETECTADA: {intencion}")

    print("🔍 RESULTADOS DE RECUPERACIÓN:\n")
    # TABULAR: tuple(DataFrame, str)
    if isinstance(resultados, tuple) and isinstance(resultados[0], pd.DataFrame):
        # Imprimir la tabla en formato Markdown para que sea legible
        print(resultados[1])
        print(resultados[0].to_markdown(index=False))

    # GRAFOS: tuple(list, str)
    elif isinstance(resultados, tuple) and isinstance(resultados[0], list):
        print(resultados[1])
        for i, res in enumerate(resultados[0][:3], 1):
            print(f"[GRAFO {i}]: {res}")

    # VECTORIAL: list[tuple(texto, score)]
    elif isinstance(resultados, list) and isinstance(resultados[0], tuple) and len(resultados) > 0:
        for i, (texto, score) in enumerate(resultados[:3], 1):
            print(f"[FRAGMENTO {i}] Score ReRank: {score:.4f}")
            print(f"Contenido: {texto[:200]}...")

    # FALLBACKS: strings de error o mensajes no reconocidos
    else:
        print(f"{resultados}")
    print("\n"+("-"*80)+"\n")

🗣️ USUARIO: '¿Cómo uso mi licuadora para hacer smoothies?'
🎯 INTENCIÓN DETECTADA: vectorial
🔍 RESULTADOS DE RECUPERACIÓN:

[FRAGMENTO 1] Score ReRank: 0.4979
Contenido: Producto: Compacto Licuadora P0004 Marca: ChefMaster | Manual Técnico - Compacto Licuadora > Procedimientos de Uso > PROCEDIMIENTO 1: Preparar Smoothie de Frutas | Dificultad: Fácil | Tiempo: 3-5 minu...
[FRAGMENTO 2] Score ReRank: 10.9654
Contenido: Reseña de usuario (1/5 estrellas) para el producto Licuadora P0007: Hola a todos! Lamentablemente debo decir que llegó defectuoso con Licuadora. Es muy voluminoso y lento. No funciona como indica la p...
[FRAGMENTO 3] Score ReRank: 10.6754
Contenido: Reseña de usuario (2/5 estrellas) para el producto Licuadora P0007: Buenas! Lamentablemente debo decir que la calidad es muy mala con Licuadora. Es muy voluminoso y lento. No funciona como indica la p...

--------------------------------------------------------------------------------

🗣️ USUARIO: '¿Cuáles son las licuadoras de

##  Generación y Conversación

In [ ]:
class AsistenteVirtual:
    def __init__(self):
        """Inicializa el chat con memoria usando LangChain y el modelo local"""
        # Instanciar modelo local
        self.llm = ChatOllama(model="qwen2.5", temperature=0.0)

        # Prompt
        prompt_sistema = """Eres un asistente experto de una empresa de electrodomésticos.
        Recuerda toda la conversación.
        Responde SIEMPRE en el mismo idioma en el que te habla el usuario.
        Basa tus respuestas ÚNICAMENTE en el contexto de la base de datos que se te proporciona en cada turno.
        Si el contexto indica que no hay datos, dile amablemente al usuario que no encontraste información e invítalo a reformular."""

        # Inicializar historial
        self.historial = [SystemMessage(content=prompt_sistema)]

    def formatear(self, resultados) -> str:
        """Convierte los resultados de las bases de datos en texto puro para el LLM"""
        # TABULAR: tuple(DataFrame, str)
        if isinstance(resultados, tuple) and isinstance(resultados[0], pd.DataFrame):
            if resultados[0].empty:
                return "No hay datos en la base de datos para esta consulta."
            return resultados[0].to_markdown(index=False)

        # GRAFOS: tuple(list, str)
        elif isinstance(resultados, tuple) and isinstance(resultados[0], list):
            if not resultados[0] or (isinstance(resultados[0][0], dict) and "Error" in str(resultados[0][0])):
                return "No hay datos en el grafo para esta consulta."
            return "\n".join([str(item) for item in resultados[0]])

        # VECTORIAL: list[tuple(texto, score)]
        elif isinstance(resultados, list) and len(resultados) > 0 and isinstance(resultados[0], tuple):
            return "\n---\n".join([texto for texto, score in resultados])

        # FALLBACKS
        return str(resultados)

    def preguntar(self, query: str) -> str:
        """Procesa la pregunta con memoria conversacional y RAG dinámico"""
        # Recuperación de contexto dinámico
        resultados, _ = pipeline_recuperacion(query)

        # Formatear resultados a texto legible para el LLM
        contexto = self.formatear(resultados)

        # Validación de datos
        if "No hay datos" in contexto or "Error" in contexto:
            response = "Lo siento, no he encontrado información relevante sobre eso en nuestra base de datos. ¿Podrías intentar reformular tu pregunta o consultar otro tema?"
            # Guardamos este intercambio en la memoria
            self.historial.append(HumanMessage(content=query))
            self.historial.append(AIMessage(content=response))

            return response

        # Armar mensaje inyectando el contexto formateado
        mensaje = f"CONTEXTO DE LA BASE DE DATOS:\n{contexto}\n\nPREGUNTA DEL USUARIO:\n{query}"

        # Agregar pregunta al historial
        self.historial.append(HumanMessage(content=mensaje))

        # Invocar el modelo con TODO el historial
        response = self.llm.invoke(self.historial)

        # Agregar respuesta al historial
        self.historial.append(AIMessage(content=response.content))

        return response.content

    def obtener_estadisticas(self):
        """Obtiene estadísticas del chat actual"""
        if len(self.historial) <= 1:
            return "No hay historial activo"

        num_mensajes = len(self.historial) - 1  # Restamos el mensaje del sistema
        num_preguntas = num_mensajes // 2

        return f"📊 ESTADÍSTICAS:\n- Mensajes en historial: {num_mensajes}\n- Preguntas realizadas: {num_preguntas}"

In [ ]:
# Instanciamos el asistente
asistente = AsistenteVirtual()

print("=== ASISTENTE VIRTUAL ESPECIALIZADO EN ELECTRODOMÉSTICOS ===")
print("Haga su consulta o enter para terminar.\n")

while True:
    user_input = input("👤 USUARIO: ")

    if user_input.strip() == "":
        print("\n🤖 ASISTENTE: ¡Hasta luego!")
        print("\n"+("-"*80)+"\n")
        print(asistente.obtener_estadisticas())
        break

    response = asistente.preguntar(user_input)
    print(f"\n🤖 ASISTENTE: {response}")
    print("\n"+("-"*80)+"\n")

=== ASISTENTE VIRTUAL ESPECIALIZADO EN ELECTRODOMÉSTICOS ===
Haga su consulta o enter para terminar.

👤 USUARIO: ¿Cómo uso mi licuadora para hacer smoothies?

🤖 ASISTENTE: Para preparar un smoothie de frutas utilizando tu compacta licuadora ChefMaster P0004, sigue estos pasos:

1. Lava y corta las frutas en trozos medianos (2-3 cm).
2. Coloca los ingredientes líquidos primero (leche, yogurt, jugo) en la jarra.
3. Agrega las frutas y hielo a continuación.
4. Cierra herméticamente la tapa de la licuadora.
5. Comienza el procesamiento en velocidad baja (niveles 1-2) durante 10 segundos.
6. Aumenta gradualmente a velocidad alta (niveles 4-5).
7. Procesa durante 45-60 segundos hasta obtener una textura homogénea.
8. Si es necesario, usa la función PULSE para romper trozos grandes.
9. Verifica la consistencia y procesa adicionalmente 10-15 segundos si es necesario.
10. Apaga y desconecta la licuadora antes de retirar la jarra.
11. Sirve inmediatamente para obtener el mejor sabor y textura.



# Ejercicio 2: Agente autónomo

## Creación de Herramientas

In [ ]:
class AsistenteVirtualReAct:
    def __init__(self, prompt_react: str):
        """Inicializa el agente autónomo ReAct con memoria usando LangChain y el modelo local"""
        # Instanciar modelo local
        self.llm = ChatOllama(model="qwen2.5", temperature=0.0)

        # Inicializar historial
        self.historial = []

        # Mapeo de herramientas
        self.herramientas = {
            "table_search": self.table_search,
            "graph_search": self.graph_search,
            "doc_search": self.doc_search,
            "analytics_tool": self.analytics_tool
        }

        self.prompt_react = prompt_react

    def doc_search(self, query: str) -> str:
        try:
            resultados = buscar_db_vectorial_hibrida(query)
            if isinstance(resultados, list) and len(resultados) > 0 and isinstance(resultados[0], tuple):
                return "\n---\n".join([texto for texto, score in resultados])
            return "No hay datos en manuales o reseñas para esta consulta."
        except Exception as e: return f"Error: {e}"

    def table_search(self, query: str) -> str:
        try:
            resultados, sql_generado = buscar_db_tabular(query)
            if isinstance(resultados, pd.DataFrame):
                if resultados.empty:
                    return f"[SQL Ejecutado: {sql_generado}]\nNo hay datos en la base de datos para esta consulta."
                return f"[SQL Ejecutado: {sql_generado}]\nDatos: {str(resultados.to_dict(orient='records'))}"
            return str(resultados)
        except Exception as e: return f"Error: {e}"

    def graph_search(self, query: str) -> str:
        try:
            resultados, cypher_generado = buscar_db_grafos(query)
            if isinstance(resultados, list):
                if not resultados or (isinstance(resultados[0], dict) and "Error" in str(resultados[0])):
                    return f"[Cypher Ejecutado: {cypher_generado}]\nNo hay datos en el grafo para esta consulta."
                return f"[Cypher Ejecutado: {cypher_generado}]\nDatos:\n" + "\n".join([str(item) for item in resultados])
            return str(resultados)
        except Exception as e: return f"Error: {e}"

    def analytics_tool(self, query: str) -> str:
        try:
            # Reutilizamos el motor tabular para que el LLM genere SQL y traiga los datos
            resultados, sql_generado = buscar_db_tabular(query)

            if isinstance(resultados, pd.DataFrame) and not resultados.empty:
                plt.figure(figsize=(10, 6))

                # Identificamos columnas numéricas
                numerics = resultados.select_dtypes(include='number')

                # CASO A: Hay al menos 2 columnas y una es numérica (Ej: GROUP BY metodo_pago, COUNT(*))
                if len(resultados.columns) >= 2 and not numerics.empty:
                    # Tomamos la primera columna categórica para el eje X, y la primera numérica para el eje Y
                    x_col = [c for c in resultados.columns if c not in numerics.columns]
                    x_col = x_col[0] if x_col else resultados.columns[0]
                    y_col = numerics.columns[0]

                    # Tomamos el Top 15 para no saturar el gráfico
                    df_plot = resultados.sort_values(by=y_col, ascending=False).head(15)

                    plt.bar(df_plot[x_col].astype(str), df_plot[y_col], color='#4C72B0', edgecolor='black')
                    plt.xticks(rotation=45, ha='right')
                    plt.xlabel(x_col.upper())
                    plt.ylabel(y_col.upper())
                    plt.title(f"{y_col.replace('_', ' ').title()} por {x_col.replace('_', ' ').title()}")

                # CASO B: Hay una sola columna numérica (Ej: SELECT precio_usd FROM productos) -> Histograma
                elif len(resultados.columns) == 1 and not numerics.empty:
                    col = numerics.columns[0]
                    plt.hist(resultados[col].dropna(), bins=20, color='#55A868', edgecolor='black')
                    plt.xlabel(col.replace('_', ' ').upper())
                    plt.ylabel("FRECUENCIA")
                    plt.title(f"Distribución de {col.replace('_', ' ').title()}")

                else:
                    plt.close()
                    return f"[SQL: {sql_generado}] Datos recuperados, pero no tienen el formato correcto (ej. falta una columna numérica para graficar)."

                plt.tight_layout()
                filename = "grafico_analiticas.png"
                plt.savefig(filename)
                plt.close()

                # Devolvemos un string al agente confirmando el éxito para que pueda armar el Final Answer
                resumen_datos = str(resultados.head(3).to_dict(orient='records'))
                return f"[SQL: {sql_generado}] ¡Éxito! Gráfico generado y guardado localmente como '{filename}'. Resumen de los datos graficados: {resumen_datos}"

            return f"[SQL: {sql_generado}] No hay datos numéricos en la base para generar el gráfico."

        except Exception as e:
            return f"Error en analytics_tool: {e}"

    def preguntar(self, query: str, max_iteraciones: int = 5) -> str:
        """Procesa la pregunta con el motor ReAct y guarda en el historial."""

        historial_reciente = "\n".join(
            [f"{'Usuario' if isinstance(m, HumanMessage) else 'Asistente'}: {m.content}" for m in self.historial[-4:]]
        ) if self.historial else "Sin historial previo."

        prompt_actual = self.prompt_react.format(historial_str=historial_reciente, input=query)
        respuesta_final = "Límite de iteraciones alcanzado sin respuesta final."

        # MEMORIA DE CORTO PLAZO ANTI-BUCLES
        acciones_ejecutadas = set()

        for paso in range(max_iteraciones):
            respuesta_llm = self.llm.invoke(
                [HumanMessage(content=prompt_actual)],
                stop=["Observation:"]
            ).content

            prompt_actual += respuesta_llm

            # Verificación de Final Answer
            match_final = re.search(r"(?:Final Answer|Respuesta Final|Respuesta)[\*]*\s*:\s*(.*)", respuesta_llm, re.IGNORECASE | re.DOTALL)
            if match_final:
                respuesta_final = match_final.group(1).strip()
                break

            # Extracción de Acción e Input
            try:
                accion_match = re.search(r"Action[\*]*\s*:\s*([^\n]*)", respuesta_llm, re.IGNORECASE)
                entrada_match = re.search(r"Action Input[\*]*\s*:\s*([^\n]*)", respuesta_llm, re.IGNORECASE)

                if not accion_match:
                    salida = re.sub(r"^(?:Thought|Pensamiento)[\*]*\s*:\s*", "", respuesta_llm.strip(), flags=re.IGNORECASE).strip()
                    if not salida:
                        prompt_actual += " (Nota interna: Ya tengo datos o la base está vacía. Debo usar 'Final Answer: ') "
                        continue
                    respuesta_final = salida
                    print(f"\n⚠️ [FALLBACK]: Se asume salida como final.")
                    break

                # Parseo y LIMPIEZA de herramientas
                accion = accion_match.group(1).strip().replace("'", "").replace('"', "")
                entrada_accion = entrada_match.group(1).strip() if entrada_match else ""

                # Eliminamos comillas literales extra que ensucian la entrada
                entrada_accion = entrada_accion.strip('\"\'')

                # Mostrar Pensamiento
                pensamiento_match = re.search(r"(.*?)Action", respuesta_llm, re.DOTALL | re.IGNORECASE)
                if pensamiento_match:
                    p = pensamiento_match.group(1).replace("Thought:", "").strip()
                    if p: print(f"\n🤔 [PENSANDO] -> {p}")

                print(f"\n🛠️ [USANDO]: '{accion}' -> INPUT: '{entrada_accion}'")

                # CONTROL DE REPETICIÓN EXACTA
                firma_accion = (accion, entrada_accion)
                if firma_accion in acciones_ejecutadas:
                    resultado_herramienta = "Error: Ya ejecutaste esta misma acción con estas mismas palabras. ¡NO REPITAS! Sintetiza la información que ya obtuviste usando 'Final Answer:' o intenta una búsqueda distinta."
                    print(f"\n🔄 [ANTI-REPETICIÓN]: El agente intentó repetir la misma acción. Fue bloqueado.")
                    # El 'continue' fue removido para permitir que el Freno de Seguridad evalúe este error.
                else:
                    acciones_ejecutadas.add(firma_accion)

                    # Ejecutar herramienta correspondiente
                    if accion in self.herramientas:
                        resultado_herramienta = self.herramientas[accion](entrada_accion)
                    else:
                        resultado_herramienta = f"Error: La herramienta '{accion}' no existe."

                res_consola = str(resultado_herramienta)
                if len(res_consola) > 500:
                    res_consola = res_consola[:500] + " ... [Texto truncado]"
                print(f"\n📄 [RESULTADOS]: {res_consola}")

                # Inyectar la observación
                prompt_actual += f"\nObservation: {resultado_herramienta}\n"

                # FRENOS DE SEGURIDAD Y NUDGES COGNITIVOS
                res_str = str(resultado_herramienta)
                if ("No hay datos" in res_str or "Error" in res_str) and paso >= 2:
                    respuesta_final = "Lo siento, después de buscar exhaustivamente, no he encontrado información que coincida exactamente con tu consulta. ¿Te gustaría intentar con otra búsqueda?"
                    print("\n🛑 [FRENO DE SEGURIDAD]: Salida forzada por falta de datos o repeticiones.")
                    break
                elif len(res_str) > 10000:
                    respuesta_final = "La búsqueda devolvió demasiada información. ¿Podrías ser un poco más específico con lo que buscas?"
                    print("\n🛑 [FRENO DE SEGURIDAD]: Exceso de información. Salida forzada.")
                    break
                elif paso >= 3:
                    respuesta_final = "He analizado varias fuentes pero la información recuperada no responde del todo tu pregunta. Por favor, intentá reformularla."
                    print("\n🛑 [FRENO DE SEGURIDAD]: Límite de razonamiento seguro (Paso 3). Cortando bucle.")
                    break
                elif "No hay datos" not in res_str and "Error" not in res_str and len(res_str) > 10:
                    # NUDGE DE ÉXITO: Si encontró datos útiles reales, lo forzamos positivamente a dar la respuesta final.
                    prompt_actual += "Thought: ¡Excelente! La herramienta me devolvió la información que necesitaba. Ahora redactaré la respuesta final al usuario basándome en estos datos.\nFinal Answer: "
                else:
                    prompt_actual += "Thought:"

            except Exception as e:
                print(f"\n⚠️ [ERROR]: {e}")
                prompt_actual += f"\nObservation: Error de formato. Usa 'Action:' y 'Action Input:'.\nThought:"

        # Guardar en la memoria oficial
        self.historial.append(HumanMessage(content=query))
        self.historial.append(AIMessage(content=respuesta_final))

        return respuesta_final

    def obtener_estadisticas(self):
        """Obtiene estadísticas del chat actual"""
        if len(self.historial) <= 1:
            return "No hay historial activo"

        num_mensajes = len(self.historial)
        num_preguntas = num_mensajes // 2

        return f"📊 ESTADÍSTICAS:\n- Mensajes en historial: {num_mensajes}\n- Preguntas realizadas: {num_preguntas}"

## Prompt del sistema del agente


In [ ]:
prompt_react = """Eres el asistente estrella de una tienda de electrodomésticos.
Tienes memoria de la conversación reciente. Para responder, usa las siguientes herramientas:

1. table_search: Busca precios, características técnicas y lista productos exactos. Entrada: consulta natural (ej: "precios de cafeteras").
2. graph_search: Úsala para buscar relaciones de categorías y preguntas frecuentes. Entrada: consulta natural (ej: "productos de categoría cocina").
3. doc_search: Busca reseñas de usuarios y manuales de instrucciones. Entrada: consulta natural (ej: "qué opinan de la cafetera").
4. analytics_tool: Genera gráficos y reportes visuales estadísticos (ej: "distribución de precios de lavarropas").

Debes usar ESTRICTAMENTE el siguiente formato:

Question: la pregunta a responder
Thought: piensa qué herramienta usar y explica tu razonamiento.
Action: la herramienta a usar, debe ser exactamente una de estas: [table_search, graph_search, doc_search, analytics_tool]
Action Input: el parámetro para la herramienta
Observation: el resultado de la herramienta
... (este ciclo Thought/Action/Observation puede repetirse N veces)
Thought: ya tengo la información suficiente.
Final Answer: la respuesta final y amigable al usuario.

REGLA ANTI-ALUCINACIONES Y REINTENTOS:
¡NO INVENTES DATOS! Si las herramientas devuelven "No hay datos", "Error" o una tabla vacía, bajo NINGUNA circunstancia inventes información.
En su lugar, ESTÁS OBLIGADO A REINTENTAR al menos una vez antes de rendirte:

* Cambia el Action Input usando palabras clave más cortas o separadas en lugar de frases largas exactas.

* Cambia de herramienta (ej: si table_search falló, busca en los manuales con doc_search).

HISTORIAL RECIENTE:
{historial_str}

Question: {input}
Thought:"""

## Prueba

In [ ]:
# Instanciamos el asistente ReAct
asistente_react = AsistenteVirtualReAct(prompt_react)

print("=== ASISTENTE VIRTUAL REACT ESPECIALIZADO EN ELECTRODOMÉSTICOS ===")
print("Haga su consulta o enter para terminar.\n")

while True:
    user_input = input("👤 USUARIO: ")

    if user_input.strip() == "":
        print("\n🤖 ASISTENTE: ¡Hasta luego!")
        print("\n"+("-"*80)+"\n")
        print(asistente_react.obtener_estadisticas())
        break

    response = asistente_react.preguntar(user_input)
    print(f"\n🤖 ASISTENTE: {response}")
    print("\n"+("-"*80)+"\n")

=== ASISTENTE VIRTUAL REACT ESPECIALIZADO EN ELECTRODOMÉSTICOS ===
Haga su consulta o enter para terminar.

👤 USUARIO: ¿Cómo uso mi licuadora para hacer smoothies?

🤔 [PENSANDO] -> Para responder a esta pregunta, necesito buscar instrucciones de usuario y posibles preguntas frecuentes relacionadas con el uso de una licuadora.

🛠️ [USANDO]: 'doc_search' -> INPUT: 'instrucciones licuadora'

📄 [RESULTADOS]: Producto: Compacto Licuadora P0004 Marca: ChefMaster | Manual Técnico - Compacto Licuadora > Especificaciones Técnicas | - Modelo: P0004
- Nombre Comercial: Compacto Licuadora
- Categoría: Cocina - Preparación
- Marca: ChefMaster
- Color: Rosa
- Potencia: 1000W
- Capacidad: 2.0L
- Voltaje: 220V
- Peso Neto: 48.7 kg
- Garantía: 24 meses
- Certificaciones: CE, RoHS, ISO 9001
- Clase Energética: A++
- Origen: Importado
---
Producto: Compacto Licuadora P0004 Marca: ChefMaster | Manual Técnico - Comp ... [Texto truncado]

⚠️ [FALLBACK]: Se asume salida como final.

🤖 ASISTENTE: Para hacer s

In [ ]:

asistente_react = AsistenteVirtualReAct(prompt_react)

print("=== ASISTENTE VIRTUAL REACT ESPECIALIZADO EN ELECTRODOMÉSTICOS ===")
print("Haga su consulta o enter para terminar.\n")

while True:
    user_input = input("👤 USUARIO: ")

    if user_input.strip() == "":
        print("\n🤖 ASISTENTE: ¡Hasta luego!")
        print("\n"+("-"*80)+"\n")
        print(asistente_react.obtener_estadisticas())
        break

    response = asistente_react.preguntar(user_input)
    print(f"\n🤖 ASISTENTE: {response}")
    print("\n"+("-"*80)+"\n")

=== ASISTENTE VIRTUAL REACT ESPECIALIZADO EN ELECTRODOMÉSTICOS ===
Haga su consulta o enter para terminar.

👤 USUARIO: ¿Qué voltaje requiere el rallador digital eléctrico?

🤔 [PENSANDO] -> pienso que necesito buscar la información sobre el voltaje del rallador digital eléctrico.

🛠️ [USANDO]: 'table_search' -> INPUT: 'voltaje requerido para rallador digital'

📄 [RESULTADOS]: [SQL Ejecutado: SELECT voltaje 
FROM productos 
WHERE nombre LIKE '%rallador digital%';]
No hay datos en la base de datos para esta consulta.

🛠️ [USANDO]: 'graph_search' -> INPUT: 'voltaje rallador digital'

📄 [RESULTADOS]: [Cypher Ejecutado: MATCH (p:Producto)-[:TIENE_FAQ]->(f:FAQ {pregunta: 'voltaje rallador digital'}) RETURN p.nombre]
No hay datos en el grafo para esta consulta.

🛠️ [USANDO]: 'doc_search' -> INPUT: 'voltaje rallador digital'

📄 [RESULTADOS]: Reseña de usuario (3/5 estrellas) para el producto Digital Rallador Eléctrico P0028: Buenas! Normal, sin sorpresas con Digital Rallador Eléctrico. Tiene co